In [1]:
# Imports
from pathlib import Path
from typing import List, Dict, Any
import os
import json
import pickle
import numpy as np

from pinecone import Pinecone, ServerlessSpec

from sklearn.feature_extraction.text import TfidfVectorizer

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from bs4 import BeautifulSoup
from langchain_openai import ChatOpenAI

# Config

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
#gemini_api_key = os.getenv("GEMINI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

In [3]:
## Define an Index name
RESUME_INDEX_NAME = "resume-hybrid-index"

#yf-idf file
TFIDF_PATH = "tfidf_vectorizer.pkl"


In [4]:
from sentence_transformers import SentenceTransformer, CrossEncoder

#Dense Model
DENSE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
dense_model = SentenceTransformer(DENSE_MODEL_NAME)


# Reranking defines
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
rerank_model = CrossEncoder(RERANK_MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [5]:
# Getting pinecone Index
pc = Pinecone(api_key=pinecone_api_key)

index = pc.Index(RESUME_INDEX_NAME)
index

Index(host='https://resume-hybrid-index-503dlyx.svc.aped-4627-b74a.pinecone.io')

In [6]:
# LLM for MqE 
# llm = ChatGoogleGenerativeAI(
#     model = "gemini-2.5-flash",
#     api_key = gemini_api_key,
# )

llm = ChatOpenAI(
    model="gpt-4.1",
    api_key= openai_api_key
)

# Step 1 : MQE (Multi Query Expansion)

In [7]:
def multi_query_ext(user_query):
    prompt = f"""
    Imagine you are helping retiver the most sutable canidates resume and you will be receving sample JD or user query about resume

    Write 3 variations based on the user question focusing different of reterival.. Do not explain what you are doing or the process

    Return only the below points
    1. The output should be sematically related to the user question
    2. Use different works compared to question but they should be related 
    3. Cover different aspects for resume filtering
    
    query_id:
    {user_query}
    """
    resp = llm.invoke(prompt)
    content = getattr(resp, "content", resp)
    
    final_query = user_query + '\n' + content
    
    return final_query

In [8]:
user_query = "Resume about Accounts"

mqe_query = multi_query_ext(user_query)
mqe_query

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

# Step 2: Hybrid Retrival

In [ ]:
# Hydbrid Retrival
def hybrid_query(query_text: str, alpha: float = 0.5, top_k: int = 8):
    """
    Hybrid search combining dense (1 - alpha) and sparse (alpha) scores.

    alpha = 0.0 => dense-only
    alpha = 1.0 => sparse-only (keyword)
    """
    alpha = float(alpha)
    alpha = max(0.0, min(1.0, alpha))

    # Load TF-IDF vectorizer trained during ingest
    with open(TFIDF_PATH, "rb") as f:
        vectorizer = pickle.load(f)

    # Dense query embedding
    q_dense = dense_model.encode([query_text], normalize_embeddings=True)[0]
    q_dense = (np.asarray(q_dense, dtype=float) * (1.0 - alpha)).tolist()

    # Sparse query vector
    q_sparse_csr = vectorizer.transform([query_text]).tocoo()
    if q_sparse_csr.nnz == 0:
        q_sparse = {"indices": [0], "values": [0.0]}
    else:
        q_sparse = {
            "indices": q_sparse_csr.col.tolist(),
            "values": (q_sparse_csr.data.astype(float) * alpha).tolist(),
        }

    # Query Pinecone
    res = index.query(
        vector=q_dense,
        sparse_vector=q_sparse,
        top_k=top_k,
        include_metadata=True,
    )

    out = []
    for m in res.get("matches", []):
        md = m.get("metadata", {}) or {}
        text = md.get("text", "") or ""
        preview = " ".join(text.split()[:120])  # short snippet

        out.append(
            {
                "id": m["id"],                 # resume_id
                "score": float(m["score"]),    # hybrid score
                "preview": preview,
                "metadata": md,
            }
        )
    return out

In [9]:
results = hybrid_query(mqe_query)
#results

NameError: name 'hybrid_query' is not defined

In [10]:
#results

# Step 3 : Re-ranking

In [11]:
# Re-ranker (Cross encode i.e., dense embeddings and BM25 Re-ranker based on sparse)
def reranker_crossencoder(query, results):
    pairs = [(query, r['preview']) for r in results]
    scores = rerank_model.predict(pairs)
    #scores = np.mod(scores)
    rescored = []
    for r, s in zip(results,scores):
        r2 = dict(r)
        r2['rerank_cross_score'] = float(s)
        rescored.append(r2)
    return sorted(rescored, key = lambda x:x['rerank_cross_score'], reverse= True)

In [12]:
rerank_results = reranker_crossencoder(mqe_query, results)
#rerank_results

NameError: name 'mqe_query' is not defined

# Step 4 : Prompt Loading & Pydantic & Rendering

In [13]:
# Prompt
# --- set the path OUTSIDE the functions ---
PROMPT_PATH = r"prompt.yaml" 

# --- tiny YAML loader (file only) ---
def load_prompt(path=PROMPT_PATH):
    """Load prompt config strictly from a YAML file."""
    from pathlib import Path
    import yaml  # pip install pyyaml

    text = Path(path).read_text(encoding="utf-8")
    cfg = yaml.safe_load(text) or {}
    
    return cfg

In [14]:
cfg = load_prompt()

# Pydantic Output parser

In [15]:
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

In [16]:
class ResumeItem(BaseModel):
    resume_id: str = Field(..., description="ID of the resume, e.g. 'Emily_Green_Resume_46'")
    filename: Optional[str] = Field(None, description="Resume file name, if present in context")
    link: Optional[str] = Field(None, description="Path/URL to the resume file, if present in context")
    jd_relevance: float = Field(..., description="Relevance to the query/JD, between 0 and 1")
    profile_summary: str = Field(..., description="2–4 line summary of this candidate vs the query")
    key_skills: List[str] = Field(default_factory=list, description="Key skills relevant to the query")
    risks_or_flags: List[str] = Field(
        default_factory=list,
        description="Any risks, gaps, or misfits vs the query. Empty if none."
    )

class ResumeResponse(BaseModel):
    query: str = Field(..., description="Original user query or JD")
    resumes: List[ResumeItem] = Field(
        ...,
        description=(
            "List of ALL resumes present in the context. "
            "Include exactly one object per resume snippet."
        )
    )

resume_parser = PydanticOutputParser(pydantic_object=ResumeResponse)

In [17]:
print(resume_parser)

pydantic_object=<class '__main__.ResumeResponse'>


In [18]:
format_instructions = resume_parser.get_format_instructions()
format_instructions

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"$defs": {"ResumeItem": {"properties": {"resume_id": {"description": "ID of the resume, e.g. \'Emily_Green_Resume_46\'", "title": "Resume Id", "type": "string"}, "filename": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "description": "Resume file name, if present in context", "title": "Filename"}, "link": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "description": "Path/URL to the resume file, if present in context", "title": "Link"}, "jd_relevance": {"description": "Relevance to the query/J

In [19]:
def render_prompt(cfg: dict, query: str, sources: str) -> str:
    """
    Render the final prompt, injecting the Pydantic format instructions.
    """
    format_instructions = resume_parser.get_format_instructions()
    vars_all = dict(
        cfg.get("vars", {}),
        query=query,
        sources=sources, 
        format_instructions=format_instructions,
    )
    return cfg["template"].format(**vars_all)

In [20]:
mqe_query

NameError: name 'mqe_query' is not defined

In [21]:
#results

In [22]:
prompt = render_prompt(cfg, mqe_query, results)

NameError: name 'mqe_query' is not defined

In [23]:
final_output = llm.invoke(prompt)

NameError: name 'prompt' is not defined

# Chains

User query --> Function(multi_query_ext) --> Retriver(hybrid_query) --> Re-ranking(reranker_crossencoder) --> Prompt template --> LLM

In [24]:
def _start(user_q):
    return {"query": user_q}

def _extract_answer(answer_obj):
    content = getattr(answer_obj, "content", answer_obj)
    return str(content).strip()

In [25]:
_start(mqe_query)

NameError: name 'mqe_query' is not defined

In [26]:
# # Wihtout doing any changes it will pass the input to next stage
# # It will always take the input as dict 
# from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# final_chain = (
#     RunnableLambda(_start)
#     | RunnableLambda(lambda x:multi_query_ext(x["query"]))
#     | RunnableLambda(lambda x:hybrid_query(x))
#     | RunnableLambda(lambda x:reranker_crossencoder(x)) # We need query which is the probal
# )

# user_query = "Reume about AI"

# #final_chain.invoke(user_query)

In [27]:
# Wihtout doing any changes it will pass the input to next stage
# It will always take the input as dict 
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableParallel

final_chain = (
    RunnableLambda(_start)
    .assign(multi_query = RunnableLambda(lambda q:multi_query_ext(q["query"])))
    .assign(results = RunnableLambda(lambda q:hybrid_query(q["multi_query"])))
    .assign(re_ranked_results = RunnableLambda(lambda q:reranker_crossencoder(q["multi_query"], q["results"])))
    .assign(prompt_temp = RunnableLambda(lambda q:load_prompt(PROMPT_PATH)))
    .assign(render_prompt = RunnableLambda(lambda q:render_prompt(q["prompt_temp"], q["multi_query"], q["re_ranked_results"])))
    .assign(output = RunnableLambda(lambda q:llm.invoke(q['render_prompt'])))
    .assign(answer_raw = RunnableLambda(lambda q:_extract_answer(q['output'])))
    .assign(
        answer_parsed=RunnableLambda(
            lambda d: resume_parser.parse(d["answer_raw"])
        )
    )
    
)

In [28]:
# Example 1
user_query = "Give me some resumes who has experience in data analytics"

final_output = final_chain.invoke(user_query)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
print(final_output["output"].content)

In [ ]:
# Example 1
user_query = "Give the results which has the highest JD relavence only one"

final_output_v1 = final_chain.invoke(user_query)

In [29]:
print(final_output_v1["output"].content)

NameError: name 'final_output_v1' is not defined

In [30]:
# Turn 1: triggers 'parse' intent -> runs final_chain -> saves to CURRENT_RESUME_JSON
demo_jd = """We are hiring for a Senior Controller / Head of Accounting role.

Requirements:
- 7-10 years of progressive experience in accounting and financial management
- Strong experience with financial reporting, month-end closing, and budget management
- Proven team leadership and cross-functional collaboration
From the available resumes in the system, shortlist the top 5 candidates"""

final_output_v2 = final_chain.invoke(demo_jd)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
print(final_output_v2["output"].content)

In [31]:
# Turn 1: triggers 'parse' intent -> runs final_chain -> saves to CURRENT_RESUME_JSON
user_follow_ques = """Filter the resume which has worked MNC compaines"""

final_output_v2 = final_chain.invoke(user_follow_ques)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
print(final_output_v2["output"].content)

# Memory Layer

Session store --> For each session based on session id, we need to store question, retirved results, follow up question

history store --> Session we need check if we have previous question & quesitons (AIMessage, HumanMessage)

In [32]:
import json
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


# Stores resume results per session
_session_store = {}


_history_store: Dict[str, ChatMessageHistory] = {}

def get_history(session_id: str) -> ChatMessageHistory:
    if session_id not in _history_store:
        _history_store[session_id] = ChatMessageHistory()
    return _history_store[session_id]

In [33]:
# User question --> 
# llm answer --> 

# Follow up question --> Append the previous Human question with new question 
# LLM Answer --> Append with new llm answer

# (HumanMessage:
# AIMessage:
# HumanMessage:
# AIMessage:)

# QA Agent --> Will already reterived resumes form base chain (For Sure) 
# (If base chain retrived results are blank, then it new question)

In [34]:
# ------------------------------------------------------------------
# QA Prompt — uses MessagesPlaceholder to inject the history buffer
# The system message pins resume JSON as ground truth every turn.
# ------------------------------------------------------------------
qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an assistant answering questions about candidate resumes.\n"
            "You are given:\n"
            "- The initial job description or query.\n"
            "- A structured Resume JSON.\n"
            "- Conversation history.\n\n"
            "Use ONLY the Resume JSON as ground truth. "
            "If something is not present there, say you don't know.\n\n"
            "Initial query / JD:\n{initial_query}\n\n"
            "Resume JSON:\n{resume_json}"
        ),
        MessagesPlaceholder("history"),   # <-- history buffer is injected here
        ("human", "{question}"),
    ]
)

def _prep_qa_inputs(d: Dict[str, Any]) -> Dict[str, Any]:
    resume_json = d["resume_json"]
    initial_query = resume_json.get("query", "")
    return {
        "question": d["question"],
        "history": d.get("history", []),
        "resume_json": json.dumps(resume_json, ensure_ascii=False),
        "initial_query": initial_query,
    }

resume_qa_chain = (
    RunnableLambda(_prep_qa_inputs)
    | qa_prompt
    | llm
    | StrOutputParser()
)

# RunnableWithMessageHistory --> 1) Load the previous history and append new user and ai message back to history and update in the promtp history placeholder

qa_agent = RunnableWithMessageHistory(
    resume_qa_chain,
    get_history,
    input_messages_key="question",   # the user's turn
    history_messages_key="history",  # maps to MessagesPlaceholder("history")
)

C:\Users\arvin\miniconda3\Lib\site-packages\IPython\core\interactiveshell.py:3747: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [35]:
# final_chain--> User question for the first time, we need to hit this final chain (which would give me some resumes)

# User ask any follow up question --> Take the inputs from response from final_chain, and along with that we need to pass 
# follow up query to llm which would give me the results....

# Pre-requ before invoking QA chain --> O/p from final chain is blank or not... 

In [36]:
# Session id Concept
import uuid

session_id = str(uuid.uuid4())
session_id

'988c003c-68a4-4a9b-a269-34b55eea4770'

In [37]:
def handle_user_message(user_input: str, session_id: str = None):

    global _history_store

    # No results for this session yet
    if session_id not in _session_store:
        print(f"{session_id} is not present in session store, executing base chain for reterival")

        out = final_chain.invoke(user_input)  # Base chain

        parsed: ResumeResponse = out["answer_parsed"]

        # Store parsed resumes for this session
        _session_store[session_id] = parsed.model_dump()

        # Fresh chat history for this session
        _history_store[session_id] = ChatMessageHistory()

        resumes = _session_store[session_id].get("resumes", [])

        lines = [
            f"Parsed {len(resumes)} resumes for your query.\n",
            "Shortlisted resumes:\n"
        ]

        for r in resumes:

            filename = r.get("filename") or r["resume_id"]
            link = r.get("link") or ""
            rel = float(r.get("jd_relevance", 0.0))

            lines.append(
                f"- [{filename}]({link})  (relevance = {rel:.2f})"
                if link
                else f"- {filename}  (relevance = {rel:.2f})"
            )
            print(lines)

        return {
            "session_id": session_id,
            "response": "\n".join(lines)
        }

    # Session already has resumes -> use QA chain
    print(f"{session_id} is already present in session store, use QA chian for answering")
    cfg = {
        "configurable": {
            "session_id": session_id
        }
    }

    response = qa_agent.invoke(
        {
            "question": user_input,
            "resume_json": _session_store[session_id]
        },
        config=cfg
    )

    return {
        "session_id": session_id,
        "response": response
    }

In [38]:
# Turn 1: triggers 'parse' intent -> runs final_chain -> saves to CURRENT_RESUME_JSON
demo_jd = """We are hiring for a Senior Controller / Head of Accounting role.

Requirements:
- 7-10 years of progressive experience in accounting and financial management
- Strong experience with financial reporting, month-end closing, and budget management
- Proven team leadership and cross-functional collaboration
From the available resumes in the system, shortlist the top 5 candidates"""

final_output_v2 = handle_user_message(demo_jd, session_id="45cc0fd0-a865-4be3-8c57-3d3657610d48")

45cc0fd0-a865-4be3-8c57-3d3657610d48 is not present in session store, executing base chain for reterival


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
print(_session_store["45cc0fd0-a865-4be3-8c57-3d3657610d48"].get("query"))

In [39]:
print(_session_store["45cc0fd0-a865-4be3-8c57-3d3657610d48"].get("resumes"))

KeyError: '45cc0fd0-a865-4be3-8c57-3d3657610d48'

In [40]:
print(_history_store)

{}


In [41]:
# Turn 1
user_follow_ques = """Filter the resume which has worked MNC compaines"""

final_output_v2 = handle_user_message(user_follow_ques, session_id="45cc0fd0-a865-4be3-8c57-3d3657610d48")

45cc0fd0-a865-4be3-8c57-3d3657610d48 is not present in session store, executing base chain for reterival


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
print(final_output_v2)

In [42]:
# Turn 1
user_follow_ques = """Filter the resume which has worked MNC compaines"""

final_output_v2 = handle_user_message(user_follow_ques, session_id="45cc0fd0-a865-4be3-8c57-123455678")

45cc0fd0-a865-4be3-8c57-123455678 is not present in session store, executing base chain for reterival


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}